# ML-10 — Content Action Playbook



This notebook turns the validated refresh score into a human-reviewed work queue. It uses the committed top-100 sample for reproducible display; the full queue remains a generated, gitignored artifact. Recommendations are decision support, not automatic publishing instructions.

## 1. Ranked actions + reason codes



Review the queue from the top, but read the action and reason codes before opening a page. The score orders attention; it does not replace editorial judgment. IDs are deliberately omitted from the displayed paper preview.

In [1]:
from pathlib import Path

import json



import pandas as pd



candidates = [Path.cwd(), *Path.cwd().parents]

root = next(

    (

        candidate

        for candidate in candidates

        if (candidate / "outputs" / "refresh_queue_sample.csv").exists()

    ),

    None,

)



if root is not None:

    QUEUE_SOURCE = root / "outputs" / "refresh_queue_sample.csv"

    EXPORT_PATH = root / "work" / "outputs" / "action_playbook.json"

else:

    QUEUE_SOURCE = (

        "https://raw.githubusercontent.com/thany-8/content-refresh-prioritizer/"

        "main/outputs/refresh_queue_sample.csv"

    )

    EXPORT_PATH = Path.cwd() / "action_playbook.json"



queue = pd.read_csv(QUEUE_SOURCE).sort_values("final_rank")

assert queue["final_rank"].is_monotonic_increasing

assert not queue.empty



preview_columns = [

    "final_rank", "final_refresh_score", "confidence", "suggested_action",

    "final_reason_codes", "impressions_90d", "sessions_90d", "avg_position", "ctr",

]

public_preview = queue[preview_columns].head(10).copy()

public_preview["final_reason_codes"] = public_preview["final_reason_codes"].str.replace("|", ", ", regex=False)

public_preview.round({"final_refresh_score": 1, "avg_position": 1, "ctr": 2})

,final_rank,final_refresh_score,confidence,suggested_action,final_reason_codes,impressions_90d,sessions_90d,avg_position,ctr
0,1,81.6,high,refresh_and_review_ctr,"declining_with_demand, low_ctr_visible_page, l...",12834,66,6.8,0.05
1,2,81.4,high,refresh_and_review_ctr,"declining_with_demand, low_ctr_visible_page, m...",8064,23,3.8,0.07
2,3,81.4,medium,refresh_and_review_ctr,"declining_with_demand, low_ctr_visible_page, m...",2498,9,10.1,0.00
3,4,81.0,high,refresh_and_review_ctr,"declining_with_demand, low_ctr_visible_page, m...",13790,27,8.2,0.12
4,5,80.9,medium,refresh_and_review_ctr,"declining_with_demand, low_ctr_visible_page, m...",3393,5,3.6,0.09
5,6,80.8,high,refresh_and_review_ctr,"declining_with_demand, low_ctr_visible_page, m...",5811,14,6.4,0.07
6,7,80.6,high,refresh_and_review_ctr,"declining_with_demand, low_ctr_visible_page, m...",1622,20,3.1,0.12
7,8,80.4,high,refresh_and_review_ctr,"declining_with_demand, low_ctr_visible_page, m...",2621,10,12.8,0.00
8,9,80.4,medium,refresh_and_review_ctr,"declining_with_demand, low_ctr_visible_page, m...",1597,5,2.7,0.13
9,10,80.3,medium,refresh,"declining_with_demand, model_decline_risk, vis...",3867,5,27.5,0.05


## 2. Intended use and limits



**User:** a FlyRank editor or strategist planning a limited review batch. **Use:** start with high-confidence, visible pages; inspect the reason codes; then choose refresh, CTR review, engagement review, expansion, or monitoring. **Limit:** the score detects patterns associated with the observed decline proxy. It does not estimate future traffic or the causal benefit of a refresh.

In [2]:
action_priority = pd.DataFrame(

    [

        (1, "refresh_and_review_ctr", "Check intent, title/snippet alignment, and live search context."),

        (2, "refresh_and_review_engagement", "Check whether the page satisfies intent and whether tracking is complete."),

        (3, "expand_and_refresh", "Confirm a real coverage gap before adding content."),

        (4, "refresh", "Review freshness, accuracy, and business priority."),

        (5, "monitor", "Defer unless new evidence raises priority."),

    ],

    columns=["priority", "action", "human_next_step"],

)

action_priority

,priority,action,human_next_step
0,1,refresh_and_review_ctr,"Check intent, title/snippet alignment, and liv..."
1,2,refresh_and_review_engagement,Check whether the page satisfies intent and wh...
2,3,expand_and_refresh,Confirm a real coverage gap before adding cont...
3,4,refresh,"Review freshness, accuracy, and business prior..."
4,5,monitor,Defer unless new evidence raises priority.


## 3. Human review + the no-go list



Before acting, a reviewer must check the page's purpose, current search results, seasonality, conversion role, previous edits, and tracking quality. Never automate publication, deletion, redirects, client communication, or a claim that a refresh will recover traffic. A low score is not permission to ignore strategically important content.

In [3]:
review_policy = {

    "required_checks": [

        "page purpose and business priority",

        "current search-result and intent context",

        "seasonality or campaign timing",

        "conversion role and internal dependencies",

        "tracking completeness and previous edits",

    ],

    "never_automate": [

        "publish or delete content",

        "redirect URLs",

        "contact clients",

        "promise traffic recovery",

    ],

}

pd.Series(review_policy)

required_checks    [page purpose and business priority, current s...
never_automate     [publish or delete content, redirect URLs, con...
dtype: object

## 4. Monitoring / retrain triggers



Monitor the queue as a workflow, not just a model score. Re-run monthly when new approved data is available. Revalidate sooner if top-50 precision falls below 0.60, the observed decline base rate moves by more than 10 percentage points, feature missingness shifts by more than 10 points, or the client/content mix changes materially.

In [4]:
monitoring_triggers = pd.DataFrame(

    [

        ("top_50_precision", "< 0.60", "Revalidate ranking and pause automatic queue promotion."),

        ("decline_base_rate", "> 10 percentage-point shift", "Recalibrate expectations and metrics."),

        ("feature_missingness", "> 10 percentage-point shift", "Audit upstream collection and imputation."),

        ("client_or_content_mix", "material change", "Repeat grouped validation before reuse."),

        ("data_age", "> 30 days", "Regenerate the queue from approved current data."),

    ],

    columns=["signal", "trigger", "response"],

)

monitoring_triggers

,signal,trigger,response
0,top_50_precision,< 0.60,Revalidate ranking and pause automatic queue p...
1,decline_base_rate,> 10 percentage-point shift,Recalibrate expectations and metrics.
2,feature_missingness,> 10 percentage-point shift,Audit upstream collection and imputation.
3,client_or_content_mix,material change,Repeat grouped validation before reuse.
4,data_age,> 30 days,Regenerate the queue from approved current data.


## 5. Exports for the paper



Export only the action policy, monitoring triggers, and aggregate top-100 mix. The row-level queue remains a generated artifact and is not duplicated into `work/`.

In [5]:
playbook_payload = {

    "queue_scope": "committed top-100 public-safe sample",

    "rows_reviewed": int(len(queue)),

    "action_priority": action_priority.to_dict(orient="records"),

    "top_100_action_mix": {

        str(action): int(count)

        for action, count in queue["suggested_action"].value_counts().items()

    },

    "top_100_confidence_mix": {

        str(label): int(count)

        for label, count in queue["confidence"].value_counts().items()

    },

    "review_policy": review_policy,

    "monitoring_triggers": monitoring_triggers.to_dict(orient="records"),

}

EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

EXPORT_PATH.write_text(json.dumps(playbook_payload, indent=2))

print(f"Saved aggregate action playbook to {EXPORT_PATH}")

Saved aggregate action playbook to /Users/thany/Documents/Development/content-refresh-prioritizer/work/outputs/action_playbook.json


## Self-check



- [x] Every section contains reasoning and supporting code

- [x] All code cells executed in order with no errors

- [x] Displayed recommendations omit client and content identifiers

- [x] Claims use observed, measured, directional, and decision-support language

- [x] Committed under `work/notebooks/` and linked from the deployed paper